# EpidemicWatch — Phase 5: Outbreak Detection Layer

Turns the trained classifier from earlier phases into an actual **outbreak
detector** by adding a time-series anomaly-detection layer on top of its
predictions.

**Data and model:** the updated Phase 2 data (~15,000 template-generated symptom
sentences from the Kaggle *Diseases and Symptoms* dataset, **5 syndrome categories**:
Dermatological, Febrile/Systemic, Gastrointestinal, Musculoskeletal/Other, Respiratory,
split across 3 hospitals), and the **Phase 3 centralized Bio_ClinicalBERT + LoRA adapter
(augmented run, test macro-F1 0.933)**.

Phase 4's federated runs (test macro-F1 0.888-0.900) did not save model weights to disk,
so the centralized adapter is used as the classifier here. Nothing in the detection layer
depends on which classifier produced the predictions, so a federated global model could be
swapped in the same way.

**What this notebook does:**
1. Assigns synthetic dates to your existing hospital records (the source dataset
   has no real timestamps)
2. Deliberately injects an artificial outbreak: a sharp rise in one syndrome category
   at one hospital during a specific window, so we have ground truth to measure detection against
3. Runs the classifier over every record to get **predicted** syndrome counts per day per
   hospital (this is all a real system would have: predictions from raw text, not true labels)
4. Applies two classic surveillance methods, **CUSUM** and **EWMA**, to those count series
5. Measures **detection delay** and **false-positive rate** for each method, and checks how
   much classifier errors affect detection

**Read before reporting results:** dates and the outbreak are synthetic, and the text is
template-generated, so this demonstrates that the pipeline works end to end, not real-world
surveillance performance.

> Run in Colab with a GPU runtime for the classification step (Section 4).


## 0. Setup

In [ ]:
# If you see a torchao/peft ImportError: run this cell, then Runtime -> Restart session.
!pip uninstall -y -q torchao
!pip install -q -U peft transformers accelerate

import os
import glob
import json
import random
import zipfile

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---- Config ---------------------------------------------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
DATA_DIR = "epidemicwatch_data"
ADAPTER_DIR = "phase3_baseline_outputs/adapter_augmented"   # Phase 3's saved LoRA checkpoint
NUM_HOSPITALS = 3
MAX_LENGTH = 96               # same as Phase 3
BATCH_SIZE = 64               # inference only, so a larger batch is fine

TOTAL_DAYS = 90
BASELINE_WINDOW = 30          # days 0-29 used to estimate "normal" rate for detectors
OUTBREAK_CATEGORY = "Respiratory"      # must be one of the 5 Phase 2 syndrome categories
OUTBREAK_HOSPITAL = None               # None = automatically use the hospital with the most records of that category
OUTBREAK_START_DAY = 50
OUTBREAK_END_DAY = 65
OUTBREAK_RELATIVE_INCREASE = 1.0       # extra cases/day = this x the normal daily rate (1.0 = doubling)

OUTPUT_DIR = "phase5_outbreak_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Config set.")

## 1. Locate Data and Model (robust search, same pattern as earlier phases)

Searches a few directory levels deep for both the hospital shard data and the
Phase 3 adapter checkpoint, rather than assuming an exact folder name --
handles zips that extract with an extra nesting level.

In [ ]:
def find_by_probe(probe_relative_path, start=".", max_depth=3):
    for depth in range(max_depth + 1):
        pattern = os.path.join(start, *(["*"] * depth), probe_relative_path)
        matches = glob.glob(pattern)
        if matches:
            n_parts = len(probe_relative_path.split(os.sep))
            root = matches[0]
            for _ in range(n_parts):
                root = os.path.dirname(root)
            return root
    return None


def ensure_uploaded(probe_relative_path, zip_prompt):
    root = find_by_probe(probe_relative_path)
    if root is None:
        from google.colab import files
        print(zip_prompt)
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        extract_target = zip_name.rsplit(".", 1)[0]
        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall(extract_target)
        root = find_by_probe(probe_relative_path)
        if root is None:
            raise FileNotFoundError(
                f"Still couldn't find '{probe_relative_path}' after extracting '{zip_name}'. "
                f"Check the zip contents match what's expected."
            )
    return root

DATA_DIR = ensure_uploaded(
    os.path.join("non_iid", "hospital_0", "train.csv"),
    "Please upload epidemicwatch_data.zip from Phase 2.",
)
print(f"Data root: {DATA_DIR}/")

ADAPTER_DIR = ensure_uploaded(
    os.path.join("adapter_augmented", "adapter_config.json"),
    "Please upload phase3_baseline_outputs.zip from Phase 3 (need the trained adapter).",
)
ADAPTER_DIR = os.path.join(ADAPTER_DIR, "adapter_augmented")
print(f"Adapter dir: {ADAPTER_DIR}/")

## 2. Load Full Hospital Data (all splits combined, for maximum volume)

Uses ALL records per hospital (train+val+test recombined) since this phase
needs volume to build a believable daily time series, not a held-out
evaluation set -- we already did model evaluation in Phases 3-4.

In [ ]:
def load_full_hospital_df(hospital_idx, data_dir=DATA_DIR, source="non_iid"):
    parts = []
    for split in ("train", "val", "test"):
        path = os.path.join(data_dir, source, f"hospital_{hospital_idx}", f"{split}.csv")
        parts.append(pd.read_csv(path))
    return pd.concat(parts, ignore_index=True)

hospital_dfs = {h: load_full_hospital_df(h) for h in range(NUM_HOSPITALS)}
for h, df in hospital_dfs.items():
    print(f"Hospital {h}: {len(df)} records, categories: "
          f"{df['syndrome_category'].value_counts().to_dict()}")

CATEGORIES = sorted(hospital_dfs[0]["syndrome_category"].unique())
print("\nCategories:", CATEGORIES)
assert OUTBREAK_CATEGORY in CATEGORIES, f"'{OUTBREAK_CATEGORY}' not in {CATEGORIES}"
assert len(CATEGORIES) == 5, f"Expected the 5 Phase 2 syndrome categories, found {CATEGORIES}"

# Choose where the outbreak happens: the hospital with the most records of that category,
# so the baseline series is dense enough for a meaningful detector.
per_hospital_counts = {h: int((df["syndrome_category"] == OUTBREAK_CATEGORY).sum()) for h, df in hospital_dfs.items()}
print(f"\n'{OUTBREAK_CATEGORY}' records per hospital: {per_hospital_counts}")
if OUTBREAK_HOSPITAL is None:
    OUTBREAK_HOSPITAL = max(per_hospital_counts, key=per_hospital_counts.get)
print(f"Outbreak hospital: {OUTBREAK_HOSPITAL}")

## 3. Assign Synthetic Dates + Inject a Deliberate Outbreak

Every existing record gets a random day in [0, `TOTAL_DAYS`). Then we inject extra synthetic
"new case" records, resampled from real text of the outbreak category at the outbreak hospital
and concentrated in the outbreak window, on top of that baseline. The number of extra cases
per day is `OUTBREAK_RELATIVE_INCREASE` times that hospital's normal daily rate for the category,
so the outbreak is a controlled relative increase (default: a doubling) however many records the
hospital has. This gives us ground truth (we know exactly when and where the injected spike is)
to measure detection performance against.


In [ ]:
def assign_baseline_dates(df, total_days=TOTAL_DAYS, seed=SEED):
    rng = np.random.default_rng(seed)
    df = df.copy()
    df["day"] = rng.integers(0, total_days, size=len(df))
    df["is_injected"] = False
    return df

def inject_outbreak(df, hospital_idx, category, start_day, end_day, extra_per_day, seed=SEED):
    """Adds synthetic extra records of `category` at `hospital_idx`, dated
    within [start_day, end_day), by resampling (with replacement) real text
    examples of that category. Returns df with the extra rows appended."""
    rng = np.random.default_rng(seed + 1)
    source_pool = df[df["syndrome_category"] == category]
    assert len(source_pool) > 0, f"No examples of '{category}' to resample from"

    window_days = list(range(start_day, end_day))
    extra_rows = []
    for day in window_days:
        n_extra = rng.poisson(extra_per_day)
        sampled = source_pool.sample(n=n_extra, replace=True, random_state=rng.integers(0, 1_000_000))
        for _, row in sampled.iterrows():
            extra_rows.append({**row.to_dict(), "day": day, "is_injected": True})

    extra_df = pd.DataFrame(extra_rows)
    print(f"Injected {len(extra_df)} extra '{category}' cases into hospital {hospital_idx}, "
          f"days {start_day}-{end_day - 1}.")
    return pd.concat([df, extra_df], ignore_index=True)


# Normal daily rate of the outbreak category at the outbreak hospital -> size of the injected surge
normal_daily_rate = per_hospital_counts[OUTBREAK_HOSPITAL] / TOTAL_DAYS
OUTBREAK_EXTRA_CASES_PER_DAY = OUTBREAK_RELATIVE_INCREASE * normal_daily_rate
print(f"Normal rate: {normal_daily_rate:.1f} '{OUTBREAK_CATEGORY}' cases/day at hospital {OUTBREAK_HOSPITAL}; "
      f"injecting ~{OUTBREAK_EXTRA_CASES_PER_DAY:.1f} extra/day for days {OUTBREAK_START_DAY}-{OUTBREAK_END_DAY - 1}.")

timestamped_hospital_dfs = {}
for h, df in hospital_dfs.items():
    dated = assign_baseline_dates(df, seed=SEED + h)  # different seed per hospital -> independent baselines
    if h == OUTBREAK_HOSPITAL:
        dated = inject_outbreak(
            dated, h, OUTBREAK_CATEGORY, OUTBREAK_START_DAY, OUTBREAK_END_DAY,
            OUTBREAK_EXTRA_CASES_PER_DAY,
        )
    timestamped_hospital_dfs[h] = dated

for h, df in timestamped_hospital_dfs.items():
    print(f"Hospital {h}: {len(df)} total records ({df['is_injected'].sum()} injected)")

## 4. Load the Classifier and Predict Every Record

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label2id = {c: i for i, c in enumerate(CATEGORIES)}
id2label = {i: c for c, i in label2id.items()}

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CATEGORIES), id2label=id2label, label2id=label2id,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.to(DEVICE)
model.eval()
print("Classifier loaded (Phase 3 centralized adapter).")

# Token-length check: make sure MAX_LENGTH does not truncate these sentences
_lens = [len(tokenizer(t)["input_ids"]) for t in hospital_dfs[0]["text"].sample(500, random_state=SEED)]
print(f"Token length: max={max(_lens)}, share over MAX_LENGTH={MAX_LENGTH}: {np.mean(np.array(_lens) > MAX_LENGTH):.2%}")


@torch.no_grad()
def predict_batch(texts, batch_size=BATCH_SIZE):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        enc = tokenizer(batch_texts, truncation=True, padding=True,
                         max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        logits = model(**enc).logits
        preds.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
    return [id2label[p] for p in preds]


for h, df in timestamped_hospital_dfs.items():
    print(f"Classifying hospital {h} ({len(df)} records)...")
    df["predicted_category"] = predict_batch(df["text"].tolist())
    # Quick sanity check: how often does the classifier's prediction match the true label?
    agreement = (df["predicted_category"] == df["syndrome_category"]).mean()
    print(f"  Prediction-vs-true-label agreement: {agreement:.3f}")

print("\nNote: this adapter was trained on ~70% of these same records (Phase 2 train split), so agreement here")
print("is higher than the held-out test score (0.933 F1) and is not a fair accuracy estimate.")


## 5. Build Daily Case-Count Time Series (from PREDICTIONS, not ground truth)

This is the realistic setup: in deployment you only ever see the classifier's
predictions on incoming text, never the true label. All detection below works
on `predicted_category` counts.

In [ ]:
def build_daily_counts(df, hospital_idx, category, total_days=TOTAL_DAYS):
    sub = df[df["predicted_category"] == category]
    counts = np.zeros(total_days)
    day_counts = sub["day"].value_counts()
    for day, n in day_counts.items():
        if 0 <= day < total_days:
            counts[day] = n
    return counts

# Time series for the actual outbreak (hospital, category)
outbreak_series = build_daily_counts(
    timestamped_hospital_dfs[OUTBREAK_HOSPITAL], OUTBREAK_HOSPITAL, OUTBREAK_CATEGORY
)

# Control series (no real outbreak): every other (hospital, category) pair, to estimate false positives
control_series = {}
for h in range(NUM_HOSPITALS):
    for cat in CATEGORIES:
        if (h, cat) == (OUTBREAK_HOSPITAL, OUTBREAK_CATEGORY):
            continue
        control_series[f"hospital_{h}_{cat}"] = build_daily_counts(timestamped_hospital_dfs[h], h, cat)

print(f"Outbreak series (hospital {OUTBREAK_HOSPITAL}, {OUTBREAK_CATEGORY}): "
      f"mean={outbreak_series.mean():.2f}, peak={outbreak_series.max():.0f}")
print(f"Built {len(control_series)} control series for false-positive checking.")

## 6. Detection Methods: CUSUM and EWMA

**CUSUM** (cumulative sum control chart) accumulates deviations above a baseline mean; once the
running sum exceeds a threshold, it signals an alarm. It is good at catching sustained, moderate shifts.

**EWMA** (exponentially weighted moving average) smooths recent counts, weighting recent days
more heavily, and compares against a control limit. It reacts quickly to sudden jumps.

Both are calibrated using the first `BASELINE_WINDOW` days only (before the outbreak starts),
which is realistic: a real system calibrates against known-normal history.

**Two design choices worth describing in your methodology:**

1. *Variance floor.* The standard deviation of a short baseline sample can be unreliable,
   especially for low-count series (some hospital/category pairs are rare under the non-IID split).
   Sigma is floored at a Poisson-motivated minimum, `sqrt(max(mean, 0.5))`, since count data has
   variance that scales with its mean.
2. *Conservative thresholds* (`h_sigma=8`, `L=5`). With 15 control series and only a 30-day baseline,
   textbook thresholds (about 5 and 3) would raise false alarms. In a simulation of Poisson counts
   (mean 3, 10 and 25 per day, 2,000 runs each), `h_sigma=8` / `L=5` gave roughly 0-3% false-alarm
   probability per 60-day monitoring window, versus about 12-17% for the textbook values, while a
   doubling lasting 15 days was still detected within about 1-7 days. Larger thresholds trade a
   little speed for fewer false alarms; adjust them if your own control results disagree.


In [ ]:
def _robust_baseline_sigma(baseline):
    """Empirical std, floored at a Poisson-motivated minimum for sparse/low-count data."""
    mu = baseline.mean()
    empirical_sigma = baseline.std()
    poisson_floor = np.sqrt(max(mu, 0.5))
    return max(empirical_sigma, poisson_floor)


def cusum_detect(counts, baseline_days=BASELINE_WINDOW, k_sigma=0.5, h_sigma=8.0):
    baseline = counts[:baseline_days]
    mu = baseline.mean()
    sigma = _robust_baseline_sigma(baseline) + 1e-9
    k, h = k_sigma * sigma, h_sigma * sigma
    S, S_trace, alarm_day = 0.0, [], None
    for t, x in enumerate(counts):
        S = max(0.0, S + (x - mu - k))
        S_trace.append(S)
        if alarm_day is None and t >= baseline_days and S > h:
            alarm_day = t
    return alarm_day, np.array(S_trace)


def ewma_detect(counts, baseline_days=BASELINE_WINDOW, lam=0.2, L=5.0):
    baseline = counts[:baseline_days]
    mu = baseline.mean()
    sigma = _robust_baseline_sigma(baseline) + 1e-9
    z, z_trace, alarm_day = mu, [], None
    for t, x in enumerate(counts):
        z = lam * x + (1 - lam) * z
        ucl = mu + L * sigma * np.sqrt((lam / (2 - lam)) * (1 - (1 - lam) ** (2 * (t + 1))))
        z_trace.append(z)
        if alarm_day is None and t >= baseline_days and z > ucl:
            alarm_day = t
    return alarm_day, np.array(z_trace)


def detect_outbreak(time_series, method="cusum", **params):
    """Reusable entry point: method in {'cusum','ewma'} -> alarm_day or None."""
    if method == "cusum":
        alarm_day, trace = cusum_detect(time_series, **params)
    elif method == "ewma":
        alarm_day, trace = ewma_detect(time_series, **params)
    else:
        raise ValueError(f"Unknown method: {method}")
    return alarm_day, trace


cusum_alarm, cusum_trace = detect_outbreak(outbreak_series, method="cusum")
ewma_alarm, ewma_trace = detect_outbreak(outbreak_series, method="ewma")

print(f"CUSUM alarm day: {cusum_alarm} "
      f"(delay after outbreak start: {cusum_alarm - OUTBREAK_START_DAY if cusum_alarm is not None else 'N/A'} days)")
print(f"EWMA alarm day: {ewma_alarm} "
      f"(delay after outbreak start: {ewma_alarm - OUTBREAK_START_DAY if ewma_alarm is not None else 'N/A'} days)")

## 7. False-Positive Check on Control Series

In [ ]:
false_positive_results = []
for name, series in control_series.items():
    fp_cusum, _ = detect_outbreak(series, method="cusum")
    fp_ewma, _ = detect_outbreak(series, method="ewma")
    false_positive_results.append({"series": name, "cusum_false_alarm_day": fp_cusum, "ewma_false_alarm_day": fp_ewma})

fp_df = pd.DataFrame(false_positive_results)
print(fp_df.to_string(index=False))

cusum_fp_rate = fp_df["cusum_false_alarm_day"].notna().mean()
ewma_fp_rate = fp_df["ewma_false_alarm_day"].notna().mean()
print(f"\nCUSUM false-positive rate on control series: {cusum_fp_rate:.2f}")
print(f"EWMA false-positive rate on control series: {ewma_fp_rate:.2f}")
print(f"Number of control series: {len(fp_df)} -> each false alarm = {1/len(fp_df):.2f} of the rate")


## 8. Plot: Case Counts with True Outbreak Window and Detected Alarms

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
days_axis = np.arange(TOTAL_DAYS)
ax.bar(days_axis, outbreak_series, color="#cccccc", label="Daily predicted case count")

ax.axvspan(OUTBREAK_START_DAY, OUTBREAK_END_DAY, color="red", alpha=0.15, label="True outbreak window")

if cusum_alarm is not None:
    ax.axvline(cusum_alarm, color="blue", linestyle="--", linewidth=2,
               label=f"CUSUM alarm (day {cusum_alarm})")
if ewma_alarm is not None:
    ax.axvline(ewma_alarm, color="green", linestyle=":", linewidth=2,
               label=f"EWMA alarm (day {ewma_alarm})")

ax.axvline(BASELINE_WINDOW, color="gray", linestyle="-", linewidth=1, alpha=0.5)
ax.text(BASELINE_WINDOW + 1, ax.get_ylim()[1] * 0.9, "baseline calibration ends", fontsize=8, color="gray")

ax.set_xlabel("Day")
ax.set_ylabel(f"Predicted '{OUTBREAK_CATEGORY}' cases (Hospital {OUTBREAK_HOSPITAL})")
ax.set_title("Outbreak Detection: Predicted Case Counts Over Time")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "outbreak_detection_timeseries.png"), dpi=150)
plt.show()

## 9. Summary Table

In [ ]:
summary_rows = []
for method, alarm_day in [("CUSUM", cusum_alarm), ("EWMA", ewma_alarm)]:
    delay = (alarm_day - OUTBREAK_START_DAY) if alarm_day is not None else None
    summary_rows.append({
        "method": method,
        "detected": alarm_day is not None,
        "alarm_day": alarm_day,
        "true_outbreak_start_day": OUTBREAK_START_DAY,
        "detection_delay_days": delay,
        "false_positive_rate_on_controls": cusum_fp_rate if method == "CUSUM" else ewma_fp_rate,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(OUTPUT_DIR, "detection_summary.csv"), index=False)

print("\nInterpretation:")
print("- detection_delay_days = alarm day minus true outbreak start. Positive = alarm came after the")
print("  start (normal); negative = alarm BEFORE the start, i.e. a chance fluctuation, which is really a")
print("  false alarm rather than early warning.")
print("- A false-positive rate > 0 on the control series means that method would have signalled an")
print("  outbreak where none was injected. Tune h_sigma / L if this happens often.")


## 9b. Does Classifier Error Hurt Detection?

Repeats the detection using the **true** syndrome labels instead of the classifier's predictions.
If the alarm days are about the same, classifier errors (about 7% at the Phase 3 test score) are not
what limits detection.


In [ ]:
true_df = timestamped_hospital_dfs[OUTBREAK_HOSPITAL]
true_counts = np.zeros(TOTAL_DAYS)
for day, n in true_df[true_df["syndrome_category"] == OUTBREAK_CATEGORY]["day"].value_counts().items():
    true_counts[day] = n

rows = []
for method in ("cusum", "ewma"):
    pred_alarm, _ = detect_outbreak(outbreak_series, method=method)
    true_alarm, _ = detect_outbreak(true_counts, method=method)
    rows.append({"method": method.upper(), "alarm_day_predicted_labels": pred_alarm, "alarm_day_true_labels": true_alarm})
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nBaseline mean (days 0-{BASELINE_WINDOW - 1}): predicted={outbreak_series[:BASELINE_WINDOW].mean():.1f}/day, "
      f"true={true_counts[:BASELINE_WINDOW].mean():.1f}/day")


## 10. What to Do With This

- If both methods detect the injected outbreak a few days after it starts, with zero or few
  false alarms on the control series, that is a clean, reportable result: the full pipeline
  (LoRA classifier -> daily aggregation -> statistical surveillance) works end to end on this
  synthetic benchmark.
- If detection is late or missed, increase `OUTBREAK_RELATIVE_INCREASE` (a stronger signal) or
  loosen `h_sigma` / `L` (more sensitive, with more false alarms; a real tradeoff worth discussing).
  A smaller `OUTBREAK_RELATIVE_INCREASE` (such as 0.3-0.5) is a harder, more realistic test.
- Results come from one random date assignment and one injected outbreak. For a stronger claim,
  repeat with different `SEED` values and report how often each method detects and how many days it takes.
- `detect_outbreak()` is reusable: pass any daily count series with `method="cusum"` or `"ewma"`.
  This is the piece a later live dashboard would call on real incoming data.
- The detection layer is classifier-agnostic. To use a federated model instead, Phase 4 would need to save
  its global adapter weights; the rest of this notebook would not change.
